# Week 8: Decoder‑Only Generation (GPT)

This notebook has two goals:

1. **Architecture intuition (PyTorch):** implement a tiny decoder‑only Transformer (causal self‑attention) to understand the forward pass + masking + next‑token loss.
2. **Power of pretraining (Hugging Face):** fine‑tune a small pretrained GPT model on a small dataset slice with **CPU‑friendly defaults**.

Notes:
- First run will download a dataset/model from Hugging Face.
- CPU training is intentionally small (few steps, short sequence length) so it finishes in a reasonable time.

In [1]:
import torch
from torch.utils.data import DataLoader, TensorDataset

from utils import TinyGPT, evaluate_tiny_gpt, get_device, train_tiny_gpt, clean_memory

seed = 204

device = get_device()
print(device)
torch.mps.manual_seed(seed)

mps


## Part A — Tiny decoder‑only Transformer in pure PyTorch

The tiny model in `utils.py` follows the same high-level decoder-only pattern as GPT:

1. **Token + position embeddings**
   - `input_ids` have shape `(batch, seq_len)`.
   - Token embeddings produce `(batch, seq_len, d_model)`.
   - Position embeddings are learned with shape `(max_seq_len, d_model)`, where `max_seq_len` is the model context window.
   - We add positions with `self.position_embedding(position).unsqueeze(0)` so the position tensor broadcasts across the batch.

2. **Causal self-attention**
   - Each token can attend only to itself and earlier tokens.
   - The causal mask prevents position `t` from seeing positions `> t`.
   - There is no padding mask in this Part A toy setup because every training window has the same fixed length.
   - **Note on Inference (KV Cache):** This notebook's evaluate_tiny_gpt recomputes the full forward pass over the trailing max_seq_len tokens at every generation step (query_len == key_len throughout) — there's no caching here. In production decoder-only models, generation instead passes only the newest token (query_len=1) and reuses cached keys/values from earlier steps, which is why build_causal_mask supports a key_len ≠ query_len diagonal offset — that code path just isn't exercised in this notebook.

3. **Pre-norm Transformer blocks with residual connections**
   - Each block uses the pattern `x = x + Attention(LayerNorm(x))`.
   - Then it uses `x = x + FeedForward(LayerNorm(x))`.
   - Those two `+ x` paths are the residual connections. By the time we leave the stack, `x` already contains the residual-updated hidden states.

4. **Final normalization + language-model head**
   - After all Transformer blocks, we apply `final_norm` once: `hidden_states = self.final_norm(x)`.
   - Then `lm_head` projects each hidden state to vocabulary logits: `logits = self.lm_head(hidden_states)`.
   - We do **not** add another residual connection around `final_norm`; the final norm is not an attention or feed-forward sublayer.
   - The LM head uses `bias=False`. We also apply **Weight Tying** (`self.lm_head.weight = self.embedding.weight`), where the input embedding and output projection share the exact same matrix. This mathematically acts as a reverse lookup and saves massive amounts of parameters.

5. **Next-token prediction loss**
   - The target is the same character window shifted one step to the right.
   - The model learns: given tokens up to position `t`, predict token `t + 1`.

In [2]:
# Tiny demo dataset (character-level) so everything is self-contained.
text = """
to be or not to be.
this is a tiny corpus for a tiny gpt.
we only want to demonstrate causal masking and next-token loss.
""".strip().lower()

chars = sorted(list(set(text)))
char_index_mapping = {ch: i for i, ch in enumerate(chars)}
index_char_mapping = {i: ch for ch, i in char_index_mapping.items()}

data = torch.tensor([char_index_mapping[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
max_seq_length = 64

# Each x predicts the same window shifted one token to the right.
x_all = torch.stack([data[i : i + max_seq_length] for i in range(len(data) - max_seq_length)])
y_all = torch.stack([data[i + 1 : i + max_seq_length + 1] for i in range(len(data) - max_seq_length)])
train_loader = DataLoader(TensorDataset(x_all, y_all), batch_size=16, shuffle=True)

vocab_size, len(data), len(train_loader)

(26, 121, 4)

In [3]:
# CPU-friendly tiny training loop (few epochs over a tiny next-token dataset)
clean_memory()
model = TinyGPT(
    vocab_size=vocab_size,
    d_model=128,
    num_heads=4,
    mlp_hidden_dim=4 * 128,
    num_layers=2,
    pad_id=None,
    dropout=0.1,
    max_seq_len=max_seq_length,
)
model

Memory cleaned.


TinyGPT(
  (embedding): Embedding(26, 128)
  (position_embedding): Embedding(64, 128)
  (dropout): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (self_attn_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (self_attn): MultiHeadAttention(
        (q_proj): Linear(in_features=128, out_features=128, bias=True)
        (k_proj): Linear(in_features=128, out_features=128, bias=True)
        (v_proj): Linear(in_features=128, out_features=128, bias=True)
        (out_proj): Linear(in_features=128, out_features=128, bias=True)
      )
      (ffn_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.1, inplace=False)
          (3): Linear(in_features=512, out_features=128, bias=True)
          (4): Dropout(p=0.1, inpla

### Decoding with temperature

`evaluate_tiny_gpt` generates text one token at a time. At each step it keeps only the model's last-position logits, rescales them by `temperature`, turns them into probabilities with softmax, and samples the next token with `torch.multinomial`.

If the vocabulary logits for the next token are $z_1, z_2, \ldots, z_V$ and the temperature is $T$, the sampled distribution is:

$$p_i = \frac{\exp(z_i / T)}{\sum_{j=1}^{V} \exp(z_j / T)}$$

- `temperature=1.0`: use the model's probabilities as-is.
- `temperature<1.0`: sharpen the distribution, making high-probability tokens more likely.
- `temperature>1.0`: flatten the distribution, making generation more random.
- `temperature -> 0`: Equivalent to greedy (argmax)

The sampling line is:

$$x_{t+1}^{(s)} \sim \operatorname{Categorical}(p), \quad s = 1, \ldots, \text{num\_samples}$$

In this notebook, `num_samples=1`, so the decoder chooses one next character per generation step, appends it to the context, and repeats until `max_new_tokens` characters have been generated.


In [4]:
model = train_tiny_gpt(
    model=model,
    device=device,
    train_loader=train_loader,
    epochs=1000,
    lr=2e-4,
)

prompt = "to be"
idx0 = torch.tensor([[char_index_mapping[c] for c in prompt.lower()]], dtype=torch.long).to(device)
out = evaluate_tiny_gpt(model, idx0, max_new_tokens=100, max_seq_len=max_seq_length, temperature=1.0)[0].tolist()
print("".join(index_char_mapping[i] for i in out))

epoch 1 | loss 81.4179
epoch 101 | loss 1.6524
epoch 201 | loss 1.1351
epoch 301 | loss 0.4998
epoch 401 | loss 0.2152
epoch 501 | loss 0.1668
epoch 601 | loss 0.0892
epoch 701 | loss 0.0377
epoch 801 | loss 0.0796
epoch 901 | loss 0.0388
to be or not to be.
this is a tiny corpus for a tiny gpt.
we only want to demonstrate causal masking and 


## Part B — Fine‑tuning a pretrained GPT model (Hugging Face)

We now fine‑tune a pretrained decoder‑only model to show how much better it is with transfer learning. Part A trained a tiny GPT from scratch on a toy character dataset; Part B starts from `distilgpt2`, which already knows English-like syntax, facts, and next-token patterns from pretraining. Fine-tuning only nudges those pretrained weights toward the style and distribution of our target dataset.

The fine-tuning pipeline has five stages:

1. **Load a dataset with the Hugging Face `datasets` SDK**: download WikiText-2 and inspect its train/validation/test splits.
2. **Load a tokenizer with the `transformers` SDK**: convert raw strings into GPT-2 token IDs.
3. **Build fixed-length language-model examples**: concatenate tokenized text, split it into `max_seq_length` chunks, and set `labels = input_ids` for causal language modeling.
4. **Create a pretrained model and `Trainer`**: load `distilgpt2`, configure training, evaluation, checkpointing, and batching.
5. **Train and generate**: run a short fine-tuning job, then sample text from the resulting model.

**CPU‑realistic defaults**
- use `distilgpt2`
- short `max seq length`
- small dataset slice
- `max_steps` instead of full epochs
- small batch size + gradient accumulation

**Gradient accumulation, in one sentence**: instead of updating the model after every tiny batch, we add up gradients across several tiny batches and then take one optimizer step, which behaves like a larger batch while using much less memory.

### Cell walkthrough: imports, SDKs, and dataset loading

This cell introduces the two Hugging Face SDKs used in Part B.

- `datasets.load_dataset(...)` downloads and prepares a named dataset from the Hugging Face Hub. Here, `load_dataset("wikitext", "wikitext-2-v1")` returns a `DatasetDict` with `train`, `validation`, and `test` splits. Each split behaves like a table: it has named columns, supports fast slicing/filtering/mapping, and stores metadata such as feature names and row counts.
- `AutoTokenizer` loads the tokenizer that matches `distilgpt2`. For GPT-2 models this is a byte-pair encoding tokenizer, so text is split into subword-like tokens rather than individual characters or whole words.
- `AutoModelForCausalLM` loads a pretrained decoder-only language model with a language-modeling head attached. `CausalLM` means the model predicts the next token using only previous tokens.
- `DataCollatorForLanguageModeling` prepares mini-batches for language modeling. With `mlm=False`, it uses causal next-token prediction, not BERT-style masked language modeling.
- `TrainingArguments` stores configuration such as batch size, learning rate, logging frequency, checkpointing, and evaluation schedule.
- `Trainer` is the high-level training loop. It builds the data loaders, moves batches through the model, computes loss, handles gradient accumulation, evaluates, logs, and saves checkpoints.

`hf_model_name`, `max_seq_length`, `n_train`, and `n_val` are deliberately small so the notebook can run on CPU or Apple Silicon without turning the lesson into an overnight job.

In [10]:
from datasets import load_dataset
import os
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

hf_model_name = "distilgpt2"
max_seq_length = 128

# Keep this small for CPU
n_train = 5000
n_val = 500

raw = load_dataset("Salesforce/wikitext", "wikitext-2-v1")
raw

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

### Cell walkthrough: tokenization and language-model examples

This cell turns raw WikiText strings into fixed-length examples that `distilgpt2` can train on.

1. `AutoTokenizer.from_pretrained(hf_model_name)` loads the exact tokenizer expected by `distilgpt2`. The model was pretrained with this vocabulary and tokenization scheme, so using the matching tokenizer is essential.
2. GPT-2 tokenizers often do not define a separate padding token. The line `tokenizer.pad_token = tokenizer.eos_token` reuses the end-of-sequence token for padding so batching utilities have a valid pad token if they need one.
3. `tokenize_fn` receives a batch of dataset rows and returns token IDs for the `text` column. `return_attention_mask=False` keeps the dataset compact because the later grouping step creates fixed-length chunks.
4. `raw.map(..., batched=True, remove_columns=...)` applies the tokenizer across each split. `batched=True` is faster because the tokenizer processes many examples at once. `remove_columns` drops the original text column after tokenization, leaving token columns such as `input_ids`.
5. `group_texts` concatenates many tokenized rows into one long stream and slices it into chunks of length `max_seq_length`. This matches causal language modeling: the model sees a sequence and learns to predict each next token inside that sequence.
6. `labels = input_ids.copy()` may look surprising, but `AutoModelForCausalLM` shifts labels internally. At position `t`, it compares the model's prediction against token `t + 1`.
7. The final `shuffle(...).select(...)` calls make small train/validation subsets. This keeps the example fast while preserving the shape of a real fine-tuning workflow.

In [11]:
tokenizer = AutoTokenizer.from_pretrained(hf_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    return tokenizer(examples["text"], return_attention_mask=False)

tok = raw.map(tokenize_fn, batched=True, remove_columns=raw["train"].column_names)

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = (len(concatenated["input_ids"]) // max_seq_length) * max_seq_length
    result = {
        k: [t[i : i + max_seq_length] for i in range(0, total_length, max_seq_length)]
        for k, t in concatenated.items()
    }
    return result

lm_ds = tok.map(group_texts, batched=True)

# Shuffle then take small CPU-friendly slices
train_ds = lm_ds["train"].shuffle(seed=seed).select(range(min(n_train, len(lm_ds["train"])) ))
val_ds = lm_ds["validation"].shuffle(seed=seed).select(range(min(n_val, len(lm_ds["validation"])) ))
train_ds, val_ds

(Dataset({
     features: ['input_ids'],
     num_rows: 5000
 }),
 Dataset({
     features: ['input_ids'],
     num_rows: 500
 }))

### Cell walkthrough: pretrained model, Trainer, and gradient accumulation

This cell builds the actual fine-tuning job.

- `AutoModelForCausalLM.from_pretrained(hf_model_name)` downloads pretrained `distilgpt2` weights and attaches the causal language-modeling output head.
- `model.resize_token_embeddings(len(tokenizer))` makes sure the model's token embedding matrix has the same vocabulary size as the tokenizer. This is especially important after changing or adding tokenizer special tokens.
- `model.to(device)` moves the model to the selected device from the first cell, such as CPU, CUDA, or MPS.
- `DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)` dynamically batches examples for causal LM training. With mlm=False, it builds labels itself as a clone of input_ids (masking any pad tokens to -100) — it doesn't use a pre-existing labels column if one is present. Since our chunks are all exactly max_seq_length with no padding, the collator's main job here is just converting the batched lists into tensors.
- `TrainingArguments` defines the training recipe. `max_steps=300` limits the run to 300 optimizer updates, regardless of how many full epochs that would be.

**Gradient accumulation detail**

`per_device_train_batch_size=1` means each forward/backward pass processes one sequence per device. That is memory-friendly, but a batch size of 1 can make optimization noisy. `gradient_accumulation_steps=8` tells `Trainer` to run 8 mini-batches, accumulate their gradients, and only then call `optimizer.step()`.

So the effective training batch size is approximately:

`effective_batch_size = per_device_train_batch_size * gradient_accumulation_steps * number_of_devices`

In this notebook, with one device, that is `1 * 8 * 1 = 8` sequences per optimizer update. Memory usage is close to batch size 1, while the optimizer receives gradients averaged over 8 sequences. The tradeoff is speed: one optimizer step now requires 8 forward/backward passes.

A few other important settings:

- `learning_rate=1e-5`: small updates, appropriate when nudging pretrained weights.
- `weight_decay=0.01`: mild regularization that discourages overly large weights.
- `warmup_steps=30`: gradually increases the learning rate at the beginning so training does not start with a sudden large update.
- `eval_strategy="steps"` and `eval_steps=100`: run validation every 100 optimizer steps.
- `save_strategy="steps"`, `save_steps=100`, and `save_total_limit=2`: checkpoint periodically but keep only the two most recent checkpoints.
- `fp16=torch.cuda.is_available()`: use half precision only on CUDA GPUs. CPU and MPS runs stay in regular precision.
- `dataloader_pin_memory`:pinned memory is a DataLoader/CUDA optimization about how data gets moved from CPU RAM to GPU memory.When pin_memory=True, the DataLoader allocates each batch in memory the OS guarantees won't be moved or swapped out. The GPU can direct memory access (DMA) from it, skipping that staging-buffer copy. It also enables asynchronous transfers (non_blocking=True): the copy can overlap with GPU compute instead of blocking the CPU thread. Net effect on CUDA: faster, more overlapped host→device transfers, at the cost of using non-swappable RAM (so it's not free, you're trading system memory flexibility for transfer speed).

In [14]:
torch.mps.manual_seed(seed)
model = AutoModelForCausalLM.from_pretrained(hf_model_name)
model.config.loss_type = "ForCausalLM"
# In our use case we simply reassign pad_token = eos_token so no need for the line below since no new token added 
# model.resize_token_embeddings(len(tokenizer))
model.to(device)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

out_dir = "./models/week8_distilgpt2_wikitext2"
os.makedirs(out_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=out_dir,
    max_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_steps=30,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    seed=seed,
    fp16=torch.cuda.is_available(),
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
)

print(trainer.args.device)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

mps


### Cell walkthrough: training, checkpoints, and generation

`trainer.train()` starts fine-tuning. During training, `Trainer` repeatedly builds a batch, runs the model forward, computes causal language-model loss, backpropagates gradients, accumulates gradients for 8 mini-batches, and then performs one optimizer update. Every 100 optimizer steps it evaluates on `val_ds` and saves a checkpoint under `out_dir`.

The commented checkpoint lines are useful when you already trained once. Instead of training again, you can load a saved folder such as `checkpoint-500` and continue directly to generation or evaluation.

The final generation cell works both before and after fine-tuning. It tokenizes a prompt, calls `model.generate(...)`, and decodes the resulting token IDs back to text. Sampling settings control the style of output: `do_sample=True` enables randomness, `temperature=0.9` controls how sharp or adventurous the distribution is, and `top_p=0.95` keeps sampling inside the smallest set of likely tokens whose cumulative probability is about 95%.

In [15]:
# trainer.train()

# If you already trained, point to a checkpoint folder here:
ckpt = "./models/week8_distilgpt2_wikitext2/checkpoint-500"
model = AutoModelForCausalLM.from_pretrained(ckpt).to(device)

print("Fine-tuning cell complete.")

Step,Training Loss,Validation Loss
100,4.032513,3.691653
200,3.914148,3.548090
300,3.771431,3.494947
400,3.742440,3.473980
500,3.749715,3.463609


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Fine-tuning cell complete.


In [19]:
# Generation demo (works before/after fine-tuning)
from transformers import set_seed

set_seed(seed)
prompt = "The meaning of life is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

gen = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    top_p=0.90,
    temperature=0.95,
)
print(tokenizer.decode(gen[0], skip_special_tokens=True))

The meaning of life is in the sense of a single individual who is "witted, independent and free, or who does not possess the right to live, and who does not have the right to live." The word was given in 1708 to the Roman Catholic Church , which had no political influence over the Romans , and was the subject of the Roman Catholic Church . The word was coined in 1710 to describe what was to occur in Europe and the Americas . The Roman Catholic Church did not become a religious organization until after


### Summary:

* Part A built a tiny decoder-only Transformer from scratch: token + learned position embeddings, pre-norm causal self-attention with weight tying, and a next-token loss, and showed it can memorize a toy character-level corpus in ~1000 epochs.
* Part B contrasted that with transfer learning: fine-tuning pretrained distilgpt2 on a small WikiText-2 slice (500 optimizer steps, batch size 1 with 8-step gradient accumulation) nudges an already-fluent language model toward the target distribution, in far fewer steps than training from scratch could produce comparably fluent output.

**The core takeaway: architecture from Part A explains how a GPT-style model works; pretraining in Part B explains why it works so much better in practice.**